In [ ]:
import pandas as pd
import numpy as np

# 1. Đọc dữ liệu từ file promotions.csv
promo_path = '/content/promotions.csv'
df_promo = pd.read_csv(promo_path)

print("Kích thước ban đầu của promotions:", df_promo.shape)
print("Số lượng giá trị thiếu ban đầu:")
print(df_promo.isnull().sum())

# Kiểm tra trùng lặp trên promo_id
duplicates_count = df_promo.duplicated(subset=['promo_id'], keep=False).sum()
print(f"\nSố lượng dòng bị trùng lặp promo_id: {duplicates_count}")

In [ ]:
# 2. Xử lý trùng lặp: Giữ lại dòng có đầy đủ dữ liệu nhất
df_promo['non_null_count'] = df_promo.notnull().sum(axis=1)
df_promo_sorted = df_promo.sort_values(by='non_null_count', ascending=False)

df_promo_cleaned = df_promo_sorted.drop_duplicates(subset=['promo_id'], keep='first').copy()
df_promo_cleaned = df_promo_cleaned.drop(columns=['non_null_count'])

print("Kích thước sau khi xử lý trùng khóa chính (promo_id):", df_promo_cleaned.shape)

In [ ]:
# Define schema columns for PROMOTION
promo_schema_cols = [
    'promo_id', 'promo_name', 'promo_type', 'discount_value', 
    'start_date', 'end_date', 'app_category', 'promo_channel', 
    'stackable_flag', 'min_order_value'
]

# Đảm bảo tất cả cột trong schema đều tồn tại trong DataFrame
for col in promo_schema_cols:
    if col not in df_promo_cleaned.columns:
        df_promo_cleaned[col] = np.nan

# 3. Điền khuyết dữ liệu thiếu theo các phương pháp phổ biến
for col in promo_schema_cols:
    if df_promo_cleaned[col].isnull().any():
        if pd.api.types.is_numeric_dtype(df_promo_cleaned[col]):
            median_val = df_promo_cleaned[col].median()
            if pd.isna(median_val):
                median_val = 0
            df_promo_cleaned[col] = df_promo_cleaned[col].fillna(median_val)
            print(f"- Điền khuyết cột số '{col}' bằng: {median_val}")
        elif pd.api.types.is_bool_dtype(df_promo_cleaned[col]):
            mode_val = df_promo_cleaned[col].mode()
            fill_val = mode_val[0] if not mode_val.empty else False
            df_promo_cleaned[col] = df_promo_cleaned[col].fillna(fill_val)
            print(f"- Điền khuyết cột logic '{col}' bằng: {fill_val}")
        else:
            mode_val = df_promo_cleaned[col].mode()
            fill_val = mode_val[0] if not mode_val.empty else "Unknown"
            df_promo_cleaned[col] = df_promo_cleaned[col].fillna(fill_val)
            print(f"- Điền khuyết cột phân loại '{col}' bằng: {fill_val}")

# Kiểm tra lại xem còn dữ liệu thiếu nào không
assert df_promo_cleaned[promo_schema_cols].isnull().sum().sum() == 0, "Vẫn còn trường bị thiếu!"

In [ ]:
# 4. Xuất dữ liệu sạch ra file theo đúng schema yêu cầu
promo_final = df_promo_cleaned[promo_schema_cols].copy()

promo_output_file = '/content/promotion_new.csv'
promo_final.to_csv(promo_output_file, index=False)

print(f"Đã xuất file thành công theo schema PROMOTION tại: {promo_output_file}")
print("Kích thước file mới:", promo_final.shape)
print("\nXem trước 5 dòng đầu tiên:")
display(promo_final.head())